In [ ]:
import os, json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

# ----------------------------------------------------------------------------- config
BASE = "/content/drive/MyDrive"
SRC = {"eco": f"{BASE}/jason dimensions ecological",
       "soc": f"{BASE}/jason dimensions social"}
OLLAMA_URL = "http://localhost:11434"
K_LIST = [1, 2, 3, 4, 6, 8, 10, 15, 20]
K_MAX, BASE_K = max(K_LIST), 4
N_BOOT, SEED = 5000, 2026
BATCH = 32

# per-model settings. num_ctx: nomic accepts 8192, embeddinggemma's maximum is 2048.
PRESETS = {
    "nomic-embed-text": dict(q="search_query: ", d="search_document: ", num_ctx=8192),
    "embeddinggemma":   dict(q="task: search result | query: ", d="title: none | text: ", num_ctx=2048),
}

RUNS = globals().get("RUNS") or [
    ("nomic-embed-text", False),
    ("nomic-embed-text", True),
    ("embeddinggemma", True),
    ("embeddinggemma", False),
]
TAG_OVERRIDE = {("nomic-embed-text", False): "v3",           # legacy run keeps its old folders
                **globals().get("TAG_OVERRIDE", {})}         # user-supplied overrides (defined before %run -i)
FORCE_RERUN = bool(globals().get("FORCE_RERUN", False))

PROMPTS = globals().get("final_prompts")
assert PROMPTS and len(PROMPTS) >= 2, "define final_prompts (your 20 prompts) before running"
print("prompts:", len(PROMPTS), "| runs:", RUNS)

NEUTRAL = [   # off-topic controls: any eco-vs-soc gap here is a store/corpus artifact, not a domain effect
    "football match results in the premier league", "recipe for chocolate cake with butter and sugar",
    "history of the Roman empire and its emperors", "how to train a neural network for image classification",
    "symptoms and treatment of the common cold", "rules of chess opening strategy",
    "plot summary of a classic detective novel", "how do airplanes stay in the air",
    "tips for learning a foreign language quickly", "history of the printing press",
    "how to change a flat tire on a car", "the life of a famous classical composer",
    "basic principles of personal budgeting", "how does a refrigerator work",
    "olympic swimming world records", "introduction to quantum mechanics",
    "how to brew good coffee at home", "architecture of gothic cathedrals",
    "how to write a cover letter for a job", "the history of the internet",
]


# ----------------------------------------------------------------------------- small helpers
def cosine(a, b):
    a, b = np.asarray(a, float).reshape(-1), np.asarray(b, float).reshape(-1)
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / d) if d else 0.0


def dom_of(meta):
    return str((meta or {}).get("source_file", "")).split("::")[0]


def boot_ci(x, rng, n=N_BOOT):
    x = np.asarray(pd.Series(x).dropna(), float)
    if len(x) < 2:
        return (float(x.mean()) if len(x) else np.nan), np.nan, np.nan
    m = rng.choice(x, (n, len(x))).mean(1)
    return float(x.mean()), float(np.quantile(m, .025)), float(np.quantile(m, .975))


def w_dcg(n):
    return 1.0 / np.log2(np.arange(2, n + 2))


def run_tag(model, use_prefix):
    if (model, use_prefix) in TAG_OVERRIDE:
        return TAG_OVERRIDE[(model, use_prefix)]
    return f"{model.replace('-', '_').replace(':', '_')}_{'pref' if use_prefix else 'nopref'}"


def load_chunks(folder, dom):
    items = []
    for fn in sorted(f for f in os.listdir(folder) if f.lower().endswith(".json")):
        with open(os.path.join(folder, fn), encoding="utf-8") as f:
            for c in json.load(f):
                if isinstance(c, str) and len(c.strip()) > 50:            # same filter for both domains
                    items.append((c, f"{dom}::{fn}"))
    return items


class PrefixedEmbedder(Embeddings):
    """Adds the model's query / document prefix. Queries go through base.embed_documents on purpose:
    langchain_ollama's embed_query itself calls embed_documents, so overriding both on a subclass would prefix twice."""
    def __init__(self, base, q_prefix="", d_prefix=""):
        self.base, self.qp, self.dp = base, q_prefix, d_prefix

    def embed_documents(self, texts):
        return self.base.embed_documents([self.dp + t for t in texts])

    def embed_query(self, text):
        return self.base.embed_documents([self.qp + text])[0]


# ----------------------------------------------------------------------------- one full run = one embedder config
class Pipeline:
    def __init__(self, model, use_prefix):
        assert model in PRESETS, f"add {model} to PRESETS"
        self.model, self.use_prefix = model, use_prefix
        self.tag = run_tag(model, use_prefix)
        self.label = f"{model} ({'prefix' if use_prefix else 'no prefix'})"
        self.store_dir = f"{BASE}/vectorstore_{self.tag}"
        self.out_dir = f"{BASE}/rank_k_results_{self.tag}"
        os.makedirs(self.store_dir, exist_ok=True)
        os.makedirs(self.out_dir, exist_ok=True)
        self.rng = np.random.default_rng(SEED)                  # same seed per run -> comparable bootstrap

        p = PRESETS[model]
        self.ctx = p["num_ctx"]
        base = OllamaEmbeddings(model=model, base_url=OLLAMA_URL, num_ctx=p["num_ctx"])
        self.emb = PrefixedEmbedder(base, p["q"] if use_prefix else "", p["d"] if use_prefix else "")
        try:
            self.dim = len(self.emb.embed_query("test"))
        except Exception as e:
            raise RuntimeError(f"cannot embed with '{model}'. Did you run `ollama pull {model}`? ({e})")
        print(f"\n{'=' * 90}\nRUN  {self.label}   tag={self.tag}   dim={self.dim}\n{'=' * 90}")

    # ---- stores (collection names are fixed; the folder separates runs)
    def open_store(self, name):
        return Chroma(persist_directory=f"{self.store_dir}/{name}", collection_name=name,
                      embedding_function=self.emb, collection_metadata={"hnsw:space": "cosine"})

    def embed_all(self, texts):
        vecs = []
        for i in range(0, len(texts), BATCH):
            for attempt in range(4):
                try:
                    vecs += self.emb.embed_documents(texts[i:i + BATCH])
                    break
                except Exception as e:
                    print("  retry", attempt, e)
                    time.sleep(2 + attempt)
            else:
                raise RuntimeError(f"embedding failed at batch starting {i}")
            if (i // BATCH) % 10 == 0:
                print(f"  embedded {len(vecs)}/{len(texts)}")
        return vecs

    def domain_store(self, dom):
        vs = self.open_store(f"{dom}_v3")
        if vs._collection.count() > 0:
            print(f"{dom}: loaded existing store ({vs._collection.count()} chunks)")
            return vs
        items = load_chunks(SRC[dom], dom)
        mx = max(len(t) for t, _ in items)
        print(f"{dom}: embedding {len(items)} chunks from {len({s for _, s in items})} files "
              f"(longest chunk {mx} chars) ...")
        if mx > 3.5 * self.ctx:
            print(f"  WARNING: longest chunk may exceed the {self.ctx}-token context of {self.model}; Ollama can truncate silently.")
        vecs = self.embed_all([t for t, _ in items])
        for i in range(0, len(items), 500):
            sl = slice(i, i + 500)
            vs._collection.add(ids=[f"{dom}_{j}" for j in range(i, min(i + 500, len(items)))],
                               documents=[t for t, _ in items[sl]],
                               metadatas=[{"source_file": s} for _, s in items[sl]],
                               embeddings=vecs[sl])
        return vs

    def mixed_store(self, stores):
        vs = self.open_store("mixed_v3")
        if vs._collection.count() > 0:
            print(f"mixed: loaded existing store ({vs._collection.count()} chunks)")
            return vs
        for dom, s in stores.items():                          # reuse vectors, no re-embedding
            d = s._collection.get(include=["documents", "metadatas", "embeddings"])
            vs._collection.add(ids=[f"mixed_{dom}_{i}" for i in d["ids"]], documents=d["documents"],
                               metadatas=d["metadatas"], embeddings=np.asarray(d["embeddings"]).tolist())
        return vs

    # ---- sanity checks (document path: stored vector must equal re-embedded document text)
    def check_store(self, vs, n=20):
        d = vs._collection.get(limit=300, include=["documents", "embeddings"])
        idx = random.sample(range(len(d["ids"])), min(n, len(d["ids"])))
        s = [cosine(self.emb.embed_documents([d["documents"][i]])[0], d["embeddings"][i]) for i in idx]
        return float(np.mean(s))

    def domain_counts(self, vs):
        ms = vs._collection.get(include=["metadatas"])["metadatas"]
        doms = pd.Series([dom_of(m) for m in ms]).value_counts().to_dict()
        return doms, len({(m or {}).get("source_file") for m in ms})

    # ---- retrieval
    def retrieve(self, vs, query, k=K_MAX):
        q = self.emb.embed_query(query)
        r = vs._collection.query(query_embeddings=[q], n_results=k, include=["metadatas", "embeddings"])
        embs = np.asarray(r["embeddings"][0])
        return [{"rank": i, "dom": dom_of(m), "cos": cosine(q, e)}
                for i, (m, e) in enumerate(zip(r["metadatas"][0], embs), 1)]

    # ---- Experiment A
    def run_separate(self, queries):
        rows = []
        for dom, vs in (("eco", self.eco_vs), ("soc", self.soc_vs)):
            for pid, q in enumerate(queries):
                hits = self.retrieve(vs, q)
                for K in K_LIST:
                    c = [h["cos"] for h in hits[:K]]
                    rows.append({"domain": dom, "prompt_id": pid, "K": K, "cos_mean": np.mean(c), "cos_rank1": hits[0]["cos"]})
        return pd.DataFrame(rows)

    @staticmethod
    def paired_diffs(df):
        out = {}
        for K, g in df.groupby("K"):
            e = g[g.domain == "eco"].set_index("prompt_id")["cos_mean"]
            s = g[g.domain == "soc"].set_index("prompt_id")["cos_mean"]
            out[K] = (e - s).dropna().to_numpy()
        return out

    def summarize_separate(self, df_task, df_ctrl):
        dt, dc = self.paired_diffs(df_task), self.paired_diffs(df_ctrl)
        rows = []
        for K in K_LIST:
            m, lo, hi = boot_ci(dt[K], self.rng)
            mc, loc, hic = boot_ci(dc[K], self.rng)
            corr = (self.rng.choice(dt[K], (N_BOOT, len(dt[K]))).mean(1)
                    - self.rng.choice(dc[K], (N_BOOT, len(dc[K]))).mean(1))
            rows.append({"K": K, "delta_task": m, "task_lo": lo, "task_hi": hi,
                         "delta_neutral_control": mc, "ctrl_lo": loc, "ctrl_hi": hic,
                         "delta_corrected": float(corr.mean()), "corr_lo": float(np.quantile(corr, .025)),
                         "corr_hi": float(np.quantile(corr, .975)), "P(corrected>0)": float((corr > 0).mean())})
        return pd.DataFrame(rows)

    # ---- Experiment B
    def run_mixed(self, queries):
        rows, rank_rows = [], []
        for pid, q in enumerate(queries):
            hits = self.retrieve(self.mixed_vs, q)
            dom = np.array([h["dom"] for h in hits])
            cos = np.array([h["cos"] for h in hits])
            for r, d in enumerate(dom, 1):
                rank_rows.append({"prompt_id": pid, "rank": r, "p_eco": d == "eco", "p_soc": d == "soc"})
            for K in K_LIST:
                d, c, w = dom[:K], cos[:K], w_dcg(K)
                e, s = d == "eco", d == "soc"
                rows.append({"prompt_id": pid, "K": K, "n_eco": int(e.sum()), "n_soc": int(s.sum()),
                             "eco_share": e.mean(), "eco_exposure": float((w * e).sum() / w.sum()),
                             "first_eco": (np.argmax(e) + 1) if e.any() else np.nan,
                             "first_soc": (np.argmax(s) + 1) if s.any() else np.nan,
                             "cos_eco": c[e].mean() if e.any() else np.nan,
                             "cos_soc": c[s].mean() if s.any() else np.nan})
        return pd.DataFrame(rows), pd.DataFrame(rank_rows)

    def summarize_mixed(self, df):
        rows = []
        for K, g in df.groupby("K"):
            r = {"K": K, "expected_share": self.expected_share}
            for c in ("eco_share", "eco_exposure", "first_eco", "first_soc"):
                r[c], r[c + "_lo"], r[c + "_hi"] = boot_ci(g[c], self.rng)
            r["eco_share_excess"] = r["eco_share"] - self.expected_share
            r["delta_cos_within_topK"] = float((g.cos_eco - g.cos_soc).dropna().mean())
            b = self.rng.choice(g.eco_share.to_numpy(), (N_BOOT, len(g))).mean(1)
            r["P(eco_share>expected)"] = float((b > self.expected_share).mean())
            rows.append(r)
        return pd.DataFrame(rows)

    # ---- full run
    def run(self, prompts, neutral):
        files = ("separate_summary.csv", "mixed_summary.csv", "rank_positions.csv")
        if not FORCE_RERUN and all(os.path.exists(f"{self.out_dir}/{f}") for f in files):
            print(f"results already exist in {self.out_dir} -> loading (set FORCE_RERUN=True to recompute)")
            return {"sep": pd.read_csv(f"{self.out_dir}/{files[0]}"),
                    "mix": pd.read_csv(f"{self.out_dir}/{files[1]}"),
                    "rank": pd.read_csv(f"{self.out_dir}/{files[2]}")}

        self.eco_vs, self.soc_vs = self.domain_store("eco"), self.domain_store("soc")
        self.mixed_vs = self.mixed_store({"eco": self.eco_vs, "soc": self.soc_vs})

        print("\n=== SANITY CHECKS ===")
        for nm, vs in (("eco", self.eco_vs), ("soc", self.soc_vs), ("mixed", self.mixed_vs)):
            stored_dim = len(vs._collection.get(limit=1, include=["embeddings"])["embeddings"][0])
            assert stored_dim == self.dim, f"{nm}: stored dim {stored_dim} != model dim {self.dim} (stale store from another model?)"
            print(f"{nm:6s} chunks={vs._collection.count():5d}  domains/files={self.domain_counts(vs)}  "
                  f"re-embed vs stored cosine={self.check_store(vs):.4f}   (expect ~1.0000)")
        cnt = self.domain_counts(self.mixed_vs)[0]
        assert set(cnt) <= {"eco", "soc"}, f"unknown domain tags in mixed store: {cnt}"
        self.expected_share = cnt.get("eco", 0) / max(1, cnt.get("eco", 0) + cnt.get("soc", 0))
        print(f"expected eco share of the mixed index = {self.expected_share:.3f}")

        print("\n=== EXPERIMENT A: separate corpora (task prompts + off-topic control) ===")
        df_task, df_ctrl = self.run_separate(prompts), self.run_separate(neutral)
        sum_sep = self.summarize_separate(df_task, df_ctrl)
        print(sum_sep.round(4).to_string(index=False))

        print("\n=== EXPERIMENT B: mixed corpus ===")
        df_mix, df_rank = self.run_mixed(prompts)
        sum_mix = self.summarize_mixed(df_mix)
        rank_sum = df_rank.groupby("rank")[["p_eco", "p_soc"]].mean().reset_index()
        print(sum_mix.round(4).to_string(index=False))
        print("\nP(chunk at rank r is ecological / social):")
        print(rank_sum.round(3).to_string(index=False))

        o = self.out_dir
        df_task.to_csv(f"{o}/separate_task_raw.csv", index=False)
        df_ctrl.to_csv(f"{o}/separate_control_raw.csv", index=False)
        sum_sep.to_csv(f"{o}/separate_summary.csv", index=False)
        df_mix.to_csv(f"{o}/mixed_raw.csv", index=False)
        sum_mix.to_csv(f"{o}/mixed_summary.csv", index=False)
        rank_sum.to_csv(f"{o}/rank_positions.csv", index=False)
        self.plot(sum_sep, sum_mix, rank_sum)
        print("\nSaved to:", o)
        return {"sep": sum_sep, "mix": sum_mix, "rank": rank_sum}

    def plot(self, sum_sep, sum_mix, rank_sum):
        fig, ax = plt.subplots(1, 3, figsize=(16, 4))
        ax[0].errorbar(sum_sep.K, sum_sep.delta_task, yerr=[sum_sep.delta_task - sum_sep.task_lo, sum_sep.task_hi - sum_sep.delta_task],
                       marker="o", capsize=3, label="task prompts")
        ax[0].errorbar(sum_sep.K, sum_sep.delta_neutral_control, yerr=[sum_sep.delta_neutral_control - sum_sep.ctrl_lo,
                       sum_sep.ctrl_hi - sum_sep.delta_neutral_control], marker="s", capsize=3, label="off-topic control")
        ax[0].plot(sum_sep.K, sum_sep.delta_corrected, "k--", marker="^", label="corrected (task - control)")
        ax[0].axhline(0, color="r", lw=.8); ax[0].axvline(BASE_K, color="gray", ls=":")
        ax[0].set_xlabel("K"); ax[0].set_ylabel("Δ cosine (eco − soc)"); ax[0].set_title("Separate corpora"); ax[0].legend(fontsize=7)

        ax[1].plot(sum_mix.K, sum_mix.eco_share, marker="o", label="eco share")
        ax[1].fill_between(sum_mix.K, sum_mix.eco_share_lo, sum_mix.eco_share_hi, alpha=.2)
        ax[1].plot(sum_mix.K, sum_mix.eco_exposure, marker="s", label="eco exposure (rank-weighted)")
        ax[1].axhline(self.expected_share, color="k", ls="--", label="index share (neutral)")
        ax[1].axvline(BASE_K, color="gray", ls=":"); ax[1].set_xlabel("K"); ax[1].set_title("Mixed corpus: who wins top-K")
        ax[1].legend(fontsize=7)

        ax[2].bar(rank_sum["rank"], rank_sum.p_eco, label="eco")
        ax[2].bar(rank_sum["rank"], rank_sum.p_soc, bottom=rank_sum.p_eco, label="social")
        ax[2].set_xlabel("rank position"); ax[2].set_ylabel("share of prompts"); ax[2].set_title("Domain at each rank"); ax[2].legend(fontsize=7)
        fig.suptitle(self.label)
        plt.tight_layout(); plt.savefig(f"{self.out_dir}/rank_k_figures.png", dpi=200); plt.show()


# ----------------------------------------------------------------------------- cross-run comparison
def compare(results):
    rows = []
    for lab, r in results.items():
        m, s, rk = r["mix"], r["sep"], r["rank"]
        exp = float(m.expected_share.iloc[0])
        at = lambda df, col, K: float(df.loc[df.K == K, col].iat[0])
        below = m[m.eco_share < exp]
        sig_K = 0                                               # largest K up to which corrected CI stays > 0
        for K in sorted(s.K):
            if float(s.loc[s.K == K, "corr_lo"].iat[0]) > 0:
                sig_K = K
            else:
                break
        rows.append({
            "run": lab,
            "ctrl_offset_mean": float(s.delta_neutral_control.mean()),
            "dcorr@1": at(s, "delta_corrected", 1), "dcorr@4": at(s, "delta_corrected", 4), "dcorr@10": at(s, "delta_corrected", 10),
            "dcorr_CI>0_up_to_K": sig_K,
            "eco_share@1": at(m, "eco_share", 1), "eco_share@4": at(m, "eco_share", 4),
            "eco_share@8": at(m, "eco_share", 8), "eco_share@20": at(m, "eco_share", 20),
            "eco_exposure@4": at(m, "eco_exposure", 4),
            "expected_share": exp,
            "crossover_K(eco_share<expected)": int(below.K.min()) if len(below) else None,
            "p_eco_rank1": float(rk.loc[rk["rank"] == 1, "p_eco"].iat[0]),
            "p_eco_ranks1-4": float(rk.loc[rk["rank"] <= 4, "p_eco"].mean()),
            "p_eco_ranks5-20": float(rk.loc[rk["rank"] > 4, "p_eco"].mean()),
        })
    cmp_df = pd.DataFrame(rows)
    print("\n" + "=" * 90 + "\nCROSS-RUN COMPARISON (compare patterns, NOT raw cosine magnitudes across models)\n" + "=" * 90)
    print(cmp_df.round(3).T.to_string(header=False))
    cmp_df.to_csv(f"{BASE}/rank_k_comparison.csv", index=False)

    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    for lab, r in results.items():
        ax[0].plot(r["mix"].K, r["mix"].eco_share, marker="o", label=lab)
        ax[1].plot(r["sep"].K, r["sep"].delta_corrected, marker="^", label=lab)
        ax[2].plot(r["rank"]["rank"], r["rank"].p_eco, marker=".", label=lab)
    ax[0].axhline(float(next(iter(results.values()))["mix"].expected_share.iloc[0]), color="k", ls="--")
    ax[0].set_title("Mixed: eco share vs K"); ax[0].set_xlabel("K")
    ax[1].axhline(0, color="r", lw=.8); ax[1].set_title("Separate: control-corrected Δ vs K"); ax[1].set_xlabel("K")
    ax[2].axhline(0.5, color="k", ls="--", lw=.8); ax[2].set_title("P(chunk at rank r is eco)"); ax[2].set_xlabel("rank")
    ax[0].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(f"{BASE}/rank_k_comparison.png", dpi=200); plt.show()
    print("Saved comparison to:", BASE)
    return cmp_df


# ----------------------------------------------------------------------------- main
RESULTS = {}
for model, use_prefix in RUNS:
    pipe = Pipeline(model, use_prefix)
    RESULTS[pipe.label] = pipe.run(PROMPTS, NEUTRAL)
CMP = compare(RESULTS) if len(RESULTS) > 1 else None